# Adult Income Classification

Projeto educacional de classificação binária com foco em reprodutibilidade, comparação de modelos e auditoria de fairness.


## 1. Objetivo

Prever se a renda anual pertence à classe `>50K` ou `<=50K`. Neste estudo, identificar a classe positiva (`>50K`) é mais importante do que obter apenas a maior Accuracy.


In [1]:
from functools import partial

import pandas as pd
from IPython.display import display
from fairlearn.metrics import (MetricFrame, count, equal_opportunity_difference,
    equalized_odds_difference, false_positive_rate, selection_rate)
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
    f1_score, make_scorer, precision_score, recall_score)
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


## 2. Carregamento dos dados

O Adult é carregado diretamente do OpenML. A versão 2 possui 48.842 registros, 14 variáveis explicativas e o target textual `class`.


In [2]:
adult = fetch_openml(name="adult", version=2, as_frame=True)
X = adult.data
y = adult.target

print(f"Linhas: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Target: {y.name}")


Linhas: 48842
Features: 14
Target: class


## 3. Inspeção inicial

A inspeção identifica tipos, distribuição do target e valores ausentes. As ausências categóricas serão tratadas dentro da pipeline, evitando que o pré-processamento aprenda com o teste.


In [3]:
colunas_numericas = X.select_dtypes(include=["number"]).columns.tolist()
colunas_categoricas = X.select_dtypes(exclude=["number"]).columns.tolist()
valores_ausentes = X.isna().sum()
valores_ausentes = valores_ausentes[valores_ausentes > 0]

print("Distribuição do target:")
display(y.value_counts())
print("Colunas numéricas:", colunas_numericas)
print("Colunas categóricas:", colunas_categoricas)
print("Valores ausentes:")
display(valores_ausentes)


Distribuição do target:


class
<=50K    37155
>50K     11687
Name: count, dtype: int64

Colunas numéricas: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Colunas categóricas: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
Valores ausentes:


workclass         2799
occupation        2809
native-country     857
dtype: int64

## 4. Separação treino/teste

O teste representa 20% dos dados. `stratify=y` preserva as classes e `random_state=42` torna a divisão reproduzível. O teste será usado apenas após a escolha do modelo.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Treino:", X_train.shape, y_train.shape)
print("Teste:", X_test.shape, y_test.shape)


Treino: (39073, 14) (39073,)
Teste: (9769, 14) (9769,)


## 5. Pré-processamento

As variáveis numéricas são padronizadas. Nas categóricas, ausências são preenchidas pela categoria mais frequente e depois é aplicado One-Hot Encoding. O `ColumnTransformer` permanece na pipeline para evitar data leakage.


In [5]:
tratamento_numerico = Pipeline([
    ("scaler", StandardScaler()),
])
tratamento_categorico = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessador = ColumnTransformer([
    ("numericas", tratamento_numerico, colunas_numericas),
    ("categoricas", tratamento_categorico, colunas_categoricas),
])


## 6. Baseline com DummyClassifier

O `DummyClassifier` prevê sempre a classe mais frequente e funciona como referência mínima. A baseline é calculada somente por validação cruzada nos dados de treino.


In [6]:
dummy = DummyClassifier(strategy="most_frequent")
scores_dummy = cross_val_score(dummy, X_train, y_train, cv=5, scoring="accuracy")
print(f"Accuracy média da baseline: {scores_dummy.mean():.6f}")
print(f"Desvio-padrão: {scores_dummy.std():.6f}")


Accuracy média da baseline: 0.760730


Desvio-padrão: 0.000044


## 7. Logistic Regression

A Regressão Logística é um modelo linear simples de explicar. Pré-processamento e estimador permanecem na mesma pipeline.


In [7]:
pipeline_logistica = Pipeline([
    ("preprocessador", preprocessador),
    ("modelo", LogisticRegression(max_iter=1000)),
])


## 8. Decision Tree

São avaliadas uma árvore sem controle de profundidade e outra com `max_depth=7`. A comparação entre treino e validação ajuda a identificar overfitting.


In [8]:
pipeline_arvore = Pipeline([
    ("preprocessador", preprocessador),
    ("modelo", DecisionTreeClassifier(random_state=42)),
])
pipeline_arvore_controlada = Pipeline([
    ("preprocessador", preprocessador),
    ("modelo", DecisionTreeClassifier(max_depth=7, random_state=42)),
])


## 9. Validação cruzada

Os modelos são comparados em cinco divisões usando somente `X_train` e `y_train`. Recall, Precision e F1 consideram `>50K` como classe positiva.


In [9]:
scorer_recall = make_scorer(recall_score, pos_label=">50K")
scorer_precision = make_scorer(precision_score, pos_label=">50K")
scorer_f1 = make_scorer(f1_score, pos_label=">50K")
metricas_validacao = {
    "accuracy": "accuracy", "recall": scorer_recall,
    "precision": scorer_precision, "f1": scorer_f1,
}
modelos = {
    "Logistic Regression": pipeline_logistica,
    "Decision Tree": pipeline_arvore,
    "Decision Tree controlada": pipeline_arvore_controlada,
}
resultados_validacao = {}
for nome, modelo in modelos.items():
    resultados_validacao[nome] = cross_validate(
        modelo, X_train, y_train, cv=5,
        scoring=metricas_validacao, return_train_score=True,
    )


## 10. Comparação dos modelos

A tabela resume as médias. A Accuracy de treino evidencia o overfitting da árvore sem controle.


In [10]:
comparacao = pd.DataFrame({
    nome: {
        "Accuracy treino": resultado["train_accuracy"].mean(),
        "Accuracy validação": resultado["test_accuracy"].mean(),
        "Recall validação": resultado["test_recall"].mean(),
        "Precision validação": resultado["test_precision"].mean(),
        "F1 validação": resultado["test_f1"].mean(),
    }
    for nome, resultado in resultados_validacao.items()
}).T
display(comparacao.round(4))


,Accuracy treino,Accuracy validação,Recall validação,Precision validação,F1 validação
Logistic Regression,0.8522,0.8511,0.5961,0.7320,0.6570
Decision Tree,0.9999,0.8110,0.6192,0.6021,0.6105
Decision Tree controlada,0.8584,0.8548,0.5622,0.7699,0.6494


## 11. Escolha do modelo final

A **Logistic Regression** foi escolhida porque apresentou Recall e F1 maiores que a Decision Tree controlada, além de boa estabilidade. A árvore sem controle mostrou forte overfitting. Para este estudo, reconhecer `>50K` recebeu mais peso do que ganhar poucos décimos de Accuracy.


## 12. Avaliação final

Após a escolha, a Logistic Regression é treinada em todo o treino e avaliada uma única vez no teste reservado.


In [11]:
pipeline_final = pipeline_logistica.fit(X_train, y_train)
y_pred_final = pipeline_final.predict(X_test)

accuracy_final = accuracy_score(y_test, y_pred_final)
recall_final = recall_score(y_test, y_pred_final, pos_label=">50K")
precision_final = precision_score(y_test, y_pred_final, pos_label=">50K")
f1_final = f1_score(y_test, y_pred_final, pos_label=">50K")
matriz_confusao = confusion_matrix(y_test, y_pred_final, labels=["<=50K", ">50K"])
metricas_finais = pd.Series({
    "Accuracy": accuracy_final, "Recall": recall_final,
    "Precision": precision_final, "F1": f1_final,
})
display(metricas_finais.to_frame("Resultado").round(5))
print("Matriz de confusão [[TN, FP], [FN, TP]]:")
print(matriz_confusao)


,Resultado
Accuracy,0.85239
Recall,0.58854
Precision,0.74138
F1,0.65618


Matriz de confusão [[TN, FP], [FN, TP]]:
[[6951  480]
 [ 962 1376]]


## 13. Classification report

A classe `<=50K` apresenta desempenho superior. A classe `>50K` é mais difícil para o modelo, principalmente pelo Recall mais baixo.


In [12]:
print(classification_report(
    y_test, y_pred_final, labels=["<=50K", ">50K"]
))


              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      7431
        >50K       0.74      0.59      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.76      0.78      9769
weighted avg       0.85      0.85      0.85      9769



## 14. Fairness

A auditoria usa `MetricFrame` para observar Accuracy, Recall, FPR, Selection Rate e quantidade por `sex` e `race`. Neste experimento, `sex` e `race` permanecem entre as features utilizadas pelo modelo e também são usados como atributos sensíveis na auditoria. Remover diretamente esses atributos não garantiria, por si só, a eliminação de possíveis vieses. `difference()` e `ratio()` resumem as distâncias das métricas de desempenho entre grupos; `count` permanece apenas na visão por grupo, pois representa tamanho da amostra. As métricas descrevem disparidades; não provam, por si sós, discriminação nem garantem justiça.


In [13]:
recall_50k = partial(recall_score, pos_label=">50K")
fpr_50k = partial(false_positive_rate, pos_label=">50K")
selection_50k = partial(selection_rate, pos_label=">50K")
metricas_fairness = {
    "accuracy": accuracy_score, "recall": recall_50k,
    "fpr": fpr_50k, "selection_rate": selection_50k, "count": count,
}
fairness_sex = MetricFrame(
    metrics=metricas_fairness, y_true=y_test, y_pred=y_pred_final,
    sensitive_features=X_test["sex"],
)
fairness_race = MetricFrame(
    metrics=metricas_fairness, y_true=y_test, y_pred=y_pred_final,
    sensitive_features=X_test["race"],
)
metricas_desempenho = ["accuracy", "recall", "fpr", "selection_rate"]
print("Métricas por sex:")
display(fairness_sex.by_group.round(4))
print("Diferenças por sex:")
display(fairness_sex.difference()[metricas_desempenho].round(4))
print("Razões por sex:")
display(fairness_sex.ratio()[metricas_desempenho].round(4))
print("Métricas por race:")
display(fairness_race.by_group.round(4))
print("Diferenças por race:")
display(fairness_race.difference()[metricas_desempenho].round(4))
print("Razões por race:")
display(fairness_race.ratio()[metricas_desempenho].round(4))


Métricas por sex:


,accuracy,recall,fpr,selection_rate,count
sex,,,,,
Female,0.9257,0.5069,0.0221,0.0758,3259.0
Male,0.8157,0.6034,0.0918,0.2472,6510.0


Diferenças por sex:


accuracy          0.1101
recall            0.0965
fpr               0.0697
selection_rate    0.1714
dtype: float64

Razões por sex:


...

,accuracy,recall,fpr,selection_rate,count
race,,,,,
Amer-Indian-Eskimo,0.8958,0.3077,0.0120,0.0521,96.0
Asian-Pac-Islander,0.8264,0.6220,0.1004,0.2379,311.0
Black,0.9081,0.4574,0.0226,0.0806,968.0
Other,0.8902,0.3000,0.0278,0.0610,82.0
White,0.8460,0.5984,0.0701,0.2038,8312.0


Diferenças por race:


accuracy          0.0817
recall            0.3220
fpr               0.0884
selection_rate    0.1859
dtype: float64

Razões por race:


accuracy          0.9100
recall            0.4824
fpr               0.1200
selection_rate    0.2189
dtype: float64

In [14]:
y_test_binario = (y_test == ">50K").astype(int)
y_pred_binario = (y_pred_final == ">50K").astype(int)
equal_opportunity_race = equal_opportunity_difference(
    y_test_binario, y_pred_binario, sensitive_features=X_test["race"]
)
equalized_odds_race = equalized_odds_difference(
    y_test_binario, y_pred_binario, sensitive_features=X_test["race"]
)
print(f"Equal Opportunity Difference (race): {equal_opportunity_race:.6f}")
print(f"Equalized Odds Difference (race): {equalized_odds_race:.6f}")


Equal Opportunity Difference (race): 0.321951
Equalized Odds Difference (race): 0.321951


## 15. Conclusão

A Logistic Regression superou a baseline e apresentou resultado estável. No teste, alcançou aproximadamente 85% de Accuracy, 59% de Recall, 74% de Precision e 66% de F1. A escolha priorizou Recall e F1 sem alterar threshold nem fazer tuning agressivo.

A auditoria encontrou disparidades entre grupos de `sex` e `race`. Equal Opportunity Difference e Equalized Odds Difference por `race` ficaram próximos de 0,322. Isso exige investigação adicional antes de qualquer uso real.


## 16. Limitações e próximos passos

- O Recall positivo ainda é limitado e há 962 falsos negativos.
- A auditoria indica disparidades e tamanhos de amostra muito diferentes.
- O Adult é histórico e educacional; os resultados não justificam decisões reais.
- Possibilidades futuras: outros modelos, tuning moderado, calibração, threshold, feature engineering e análise mais profunda de fairness. Nada disso foi implementado aqui.
